# Co-op Rent Ledger Report — HAUS Rosalie / uRth Haus

Reads a multi-tenant ledger export (the format with a `Balance Forward` row per tenant,
followed by rent/charge/payment rows, blocks separated by blank rows) and produces an
income treasury report for one house, both houses, or a combined total, over a chosen
date range.

**Report covers:** rent billed, other charges billed, amount collected, collection rate,
and an arrears list (who owes what) as of the end of the selected period.

Run the cells top to bottom. You'll be prompted for the ledger filename, how to handle
any unrecognized property tags, the house (or both), and the date range — nothing needs
to be edited in the code itself.

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 1. Parse the ledger

Handles the quirks in this export format:
- Each tenant block starts with a `Balance Forward` row (name in the PROPERTY column, opening balance in BALANCE).
- Subsequent rows in the block carry `House, Unit` in PROPERTY (e.g. `Urth, Urth-10`) instead of the tenant name.
- Blocks are separated by a fully blank row.
- Rows that don't cleanly say `Urth` or `Rosalie` (e.g. `z_Debt_Payment`, `z_Associate_Members`) are kept with their raw tag — you'll be asked what to do with them in the next step, rather than the script guessing.

In [ ]:
def _to_float(val):
    """Parse a ledger number field ("1,849.13", "", None) into a float."""
    if val is None:
        return 0.0
    val = str(val).replace(",", "").strip()
    return float(val) if val else 0.0


def parse_ledger(filepath):
    """Turn the multi-tenant ledger CSV into one tidy row per transaction."""
    raw = pd.read_csv(filepath, dtype=str, keep_default_na=False)
    raw.columns = [c.strip() for c in raw.columns]

    records = []
    current_tenant = None

    for i, row in raw.iterrows():
        prop = row["PROPERTY"].strip()
        txn = row["TRANSACTION"].strip()
        date = row["DATE"].strip()

        # blank row = end of this tenant's block
        if not prop and not txn and not date:
            current_tenant = None
            continue

        if txn == "Balance Forward":
            current_tenant = prop
            records.append({
                "seq": i, "tenant": current_tenant, "house": None, "unit": None,
                "date": date, "transaction": txn,
                "debit": 0.0, "credit": 0.0, "balance": _to_float(row["BALANCE"]),
            })
            continue

        if "," in prop:
            house_part, unit_part = [p.strip() for p in prop.split(",", 1)]
        else:
            house_part, unit_part = prop, prop

        hl = house_part.lower()
        if "urth" in hl:
            house = "Urth"
        elif "rosalie" in hl:
            house = "Rosalie"
        else:
            house = house_part  # e.g. z_Debt_Payment, z_Associate_Members — resolved interactively later

        records.append({
            "seq": i, "tenant": current_tenant, "house": house, "unit": unit_part,
            "date": date, "transaction": txn,
            "debit": _to_float(row["DEBIT"]), "credit": _to_float(row["CREDIT"]),
            "balance": _to_float(row["BALANCE"]),
        })

    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df["date"], format="%m/%d/%Y", errors="coerce")
    return df

## 2. Resolve any unrecognized property tags

For anything not tagged `Urth` or `Rosalie` (e.g. `z_Associate_Members`), you'll be asked which house it belongs to. You can also skip a tag, which leaves those rows out of both house-specific reports.

In [ ]:
def resolve_flagged_houses(df):
    """Ask the user which house each unrecognized property tag belongs to."""
    df = df.copy()
    flagged = sorted(h for h in df["house"].dropna().unique() if h not in ("Urth", "Rosalie"))

    for tag in flagged:
        tenants = ", ".join(sorted(df.loc[df["house"] == tag, "tenant"].dropna().unique()))
        print(f"\nUnrecognized property tag: {tag!r} (tenant(s): {tenants})")
        while True:
            choice = input("  Assign to Urth, Rosalie, or skip (leave unassigned) [Urth/Rosalie/skip]: ").strip().lower()
            if choice in ("urth", "rosalie"):
                df.loc[df["house"] == tag, "house"] = choice.capitalize()
                break
            elif choice in ("skip", ""):
                print(f"  Leaving {tag!r} unassigned — excluded from house-specific reports.")
                break
            else:
                print("  Please type 'Urth', 'Rosalie', or 'skip'.")

    return df

## 3. Load the file

In [ ]:
DEFAULT_LEDGER_FILE = "Multi-Tenant_Ledger__2026-07-01--2026-07-31__1_.csv"

ledger_path = input(f"Ledger CSV filename [{DEFAULT_LEDGER_FILE}]: ").strip() or DEFAULT_LEDGER_FILE

df = parse_ledger(ledger_path)
print(f"Loaded {len(df)} ledger rows, {df['tenant'].nunique()} tenants, "
      f"{df['date'].min().date()} to {df['date'].max().date()}")

df = resolve_flagged_houses(df)

## 4. Choose house(s) and timeframe

In [ ]:
def choose_house(df):
    available = sorted(h for h in df["house"].dropna().unique() if h in ("Urth", "Rosalie"))
    options = available + ["Both"]
    while True:
        choice = input(f"Choose house ({'/'.join(options)}): ").strip().lower()
        for opt in options:
            if choice == opt.lower():
                return opt
        print(f"Please type one of: {', '.join(options)}")


def choose_timeframe(df):
    min_d, max_d = df["date"].min(), df["date"].max()
    print(f"Data available from {min_d.date()} to {max_d.date()}")
    start_in = input(f"Start date (YYYY-MM-DD) [{min_d.date()}]: ").strip()
    end_in = input(f"End date (YYYY-MM-DD) [{max_d.date()}]: ").strip()
    start = pd.to_datetime(start_in) if start_in else min_d
    end = pd.to_datetime(end_in) if end_in else max_d
    return start, end

## 5. Build the report

In [ ]:
def generate_report(df, house, start, end):
    in_house = df["house"] == house
    in_period = (df["date"] >= start) & (df["date"] <= end)
    period = df[in_house & in_period]

    rent_billed = period.loc[period["transaction"].str.contains("Rent", case=False, na=False), "debit"].sum()
    other_charges = period.loc[period["transaction"].isin(["110 Charge", "Late fee"]), "debit"].sum()
    collected = period.loc[period["transaction"].str.contains("Rental Income", case=False, na=False), "credit"].sum()
    total_billed = rent_billed + other_charges
    collection_rate = collected / total_billed if total_billed else float("nan")

    # ending balance per tenant in this house, as of end date (last txn on/before `end`, in original ledger order)
    house_to_date = df[(df["house"] == house) & (df["date"] <= end)].sort_values(["date", "seq"], kind="stable")
    ending = house_to_date.groupby("tenant", sort=False).tail(1)[["tenant", "unit", "balance"]]
    arrears = ending[ending["balance"] < 0].sort_values("balance")
    credit_balances = ending[ending["balance"] > 0].sort_values("balance", ascending=False)

    print(f"{house} — Income Report: {start.date()} to {end.date()}")
    print("=" * 52)
    print(f"Rent billed:      {rent_billed:>12,.2f}")
    print(f"Other charges:    {other_charges:>12,.2f}")
    print(f"Total billed:     {total_billed:>12,.2f}")
    print(f"Collected:        {collected:>12,.2f}")
    rate_str = f"{collection_rate:.1%}" if total_billed else "n/a"
    print(f"Collection rate:  {rate_str:>12}")

    print(f"\nArrears as of {end.date()}:")
    if arrears.empty:
        print("  None — all accounts current.")
    else:
        for _, r in arrears.iterrows():
            print(f"  {r['tenant']:<25} {str(r['unit']):<14} owes {-r['balance']:>10,.2f}")
        print(f"  {'TOTAL OUTSTANDING':<40} {-arrears['balance'].sum():>10,.2f}")

    print(f"\nCredit balances as of {end.date()}:")
    if credit_balances.empty:
        print("  None.")
    else:
        for _, r in credit_balances.iterrows():
            print(f"  {r['tenant']:<25} {str(r['unit']):<14} credit {r['balance']:>9,.2f}")

    return {
        "house": house, "start": start, "end": end,
        "rent_billed": rent_billed, "other_charges": other_charges,
        "total_billed": total_billed, "collected": collected,
        "collection_rate": collection_rate,
        "arrears": arrears, "credit_balances": credit_balances,
        "transactions": period,
    }


def run_report(df, house_choice, start, end):
    """Run generate_report for one house, or for both with a combined summary."""
    if house_choice == "Both":
        houses = sorted(h for h in df["house"].dropna().unique() if h in ("Urth", "Rosalie"))
    else:
        houses = [house_choice]

    reports = {}
    for h in houses:
        reports[h] = generate_report(df, h, start, end)
        print()

    if len(houses) > 1:
        combined_billed = sum(r["total_billed"] for r in reports.values())
        combined_collected = sum(r["collected"] for r in reports.values())
        combined_arrears = sum(-r["arrears"]["balance"].sum() for r in reports.values())
        combined_rate = combined_collected / combined_billed if combined_billed else float("nan")

        print("Combined — All Houses")
        print("=" * 52)
        print(f"Total billed:     {combined_billed:>12,.2f}")
        print(f"Collected:        {combined_collected:>12,.2f}")
        print(f"Collection rate:  {combined_rate:>12.1%}")
        print(f"Total arrears:    {combined_arrears:>12,.2f}")

    return reports

## 6. Run it

Prompts you for the house (or both) and the date range, then prints the report(s).

In [ ]:
house_choice = choose_house(df)
start, end = choose_timeframe(df)
reports = run_report(df, house_choice, start, end)

## 7. Optional — export to CSV

In [ ]:
export = input("Export arrears + summary to CSV? (y/n): ").strip().lower()
if export == "y":
    for house_name, report in reports.items():
        out_name = f"{report['house']}_income_report_{report['start'].date()}_{report['end'].date()}.csv"
        summary_rows = [
            {"item": "Rent billed", "amount": report["rent_billed"]},
            {"item": "Other charges", "amount": report["other_charges"]},
            {"item": "Total billed", "amount": report["total_billed"]},
            {"item": "Collected", "amount": report["collected"]},
            {"item": "Collection rate", "amount": report["collection_rate"]},
        ]
        with open(out_name, "w") as f:
            f.write("SUMMARY\n")
            pd.DataFrame(summary_rows).to_csv(f, index=False)
            f.write("\nARREARS\n")
            report["arrears"].assign(owed=lambda d: -d["balance"]).drop(columns="balance").to_csv(f, index=False)
        print(f"Saved {out_name}")